# House Price Prediction — Advanced Regression 
**Goal:** Predict the sale price of homes in Ames, Iowa.  
**Metric:** RMSE on log-transformed SalePrice.

## Problem Statement

We are given a dataset containing detailed information about 1,460 residential properties in Ames Iowa. The dataset includes a wide range of features describing each house, such as its size, construction quality, age, and various structural and neighborhood. The objective is to build a model that can accurately predict the final sale price of each property based on these features.

**Imports** 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import joblib
import kagglehub
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# import missingno as msno

**DATASET** 

In [ ]:


path = "/kaggle/input/competitions/house-prices-advanced-regression-techniques"

print(os.listdir(path))

In [ ]:
train = pd.read_csv(f"{path}/train.csv")
test  = pd.read_csv(f"{path}/test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv")


#________________________________________________________________________________

# Exploratory Data Analysis
Before touching the data, I want to actually understand it.  
What's missing? What's skewed? Which features actually relate to price?  
Which ones are useless noise?

I'll go through:
- Missing values — how bad is it, and does it affect price
- Feature types — numerical vs categorical, discrete vs continuous
- Univariate plots — what does each feature look like on its own
- Bivariate plots — how does each feature relate to SalePrice
- Correlations — are any features basically saying the same thing
- The target — is SalePrice normally distributed ?

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

Show missing values

In [ ]:
train.columns[train.isnull().sum()>1]

In [ ]:
mising_values = train.isna().sum()
mising_percent = ((train.isna().sum())/len(train))*100

missing = pd.concat([mising_values , mising_percent] ,keys=['Total','Percentage'],axis = 1 )
result = missing[missing['Percentage'] > 0 ].sort_values(by='Percentage',ascending= False)

result['Percentage'] = result['Percentage'].map(lambda x : f"{x :.2f}%")

result

In [ ]:

Mising_test = test.isna().sum()
mising_perc = (test.isna().mean())

mis = pd.concat([Mising_test ,mising_perc] , keys=['Total' , 'Percentage'] , axis = 1)
result = mis[mis['Percentage'] > 0].sort_values(by ='Percentage' ,ascending= False)


result['Percentage'] = result['Percentage'].map(lambda y: f"{y *100 :.2f}%")

result




Plot the effect of Missing values


In [ ]:

# plot the missing value effect on target to see if it need flag or just imputation
feature_with_na = [feture for feture in train.columns if train[feture].isna().sum() > 1]

fig, axs = plt.subplots(5, 4, figsize=(15, 15))
fig.patch.set_facecolor('#D5DDE3')

data = train.copy()
for x, ax in zip(feature_with_na, axs.flatten()):
    data[x] = data[x].isnull().astype(int)
    data.groupby(x)['SalePrice'].median().plot.bar(ax=ax, color=['#64877D','#727C85'],
                                                    edgecolor='white', linewidth=0.5)



plt.tight_layout()
plt.show()

In [ ]:
# # missin Train
plt.figure(figsize=(18 ,7))
colours = ['#34495E', 'seagreen']
sns.heatmap(train.isnull() , cmap=sns.color_palette(colours))

**conclusion**

- ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType']  _> drop

- ['GarageQual','GarageCond','GarageFinish','GarageType','BsmtCond','BsmtQual','BsmtExposure','BsmtFinType1','BsmtFinType2'] absence and low %  -> fill with NONE

- ['GarageYrBlt','MasVnrArea,,'BsmtFullBath' , ''  ] ->  fill with 0

- '[LotFrontage']  -> 15%   doesn't affect the target _>  impute   


-
-

**Initial Data Cleaning**

## DATA type Sepration

diffrent data type require diffrent process ,plots and statics  

NUMERICAL FEATURES


In [ ]:
# numeric feature


numerical_feature = [col for col in train.columns if train[col].dtype in ['int64', 'float64']]

print (len(numerical_feature))
print (numerical_feature)

print ("__________________\n")

train[numerical_feature].head()


Years feature

In [ ]:
years_feature = [i for i in train.columns if 'Yr' in i  or 'Year' in i]

print (years_feature)

In [ ]:
train[years_feature].head()

In [ ]:
def years_mod(df):
    df['HouseAge']   = (df['YrSold'] - df['YearBuilt']).clip(lower=0)
    df['garageAge']  = (df['YrSold'] - df['GarageYrBlt']).clip(lower=0)
    df['garageAge']  = df['garageAge'].fillna(0)
    df['remodAge']   = (df['YrSold'] - df['YearRemodAdd']).clip(lower=0)
    df['Remodeled']  = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
    df['IsNew']      = (df['YrSold'] == df['YearBuilt']).astype(int)

    return df

train = years_mod(train)
test  = years_mod(test)

In [ ]:
# distrbution of age features

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, feat in zip(axes, ['HouseAge', 'remodAge', 'garageAge']):
    sns.histplot(x=feat, data=train, ax=ax, color='steelblue', kde=True)
    ax.set_title(f'{feat} Distribution')

plt.tight_layout()
plt.show()

Discrete feature

In [ ]:
discrete_feature =  [feature for feature in numerical_feature if len(train[feature].unique())<15 and feature not in years_feature]


print (len(discrete_feature))
print (discrete_feature)

In [ ]:
train[discrete_feature].head()

In [ ]:
# plot the descrete feature for Univariate analysis
fig, ax = plt.subplots(4, 4, figsize=(20, 20))
for feature, axi in zip(discrete_feature, ax.flatten()):

  sns.countplot(x = feature ,y = None ,data = train ,  ax =axi)


- OverallQual Bin rare levels -> bin (1,2 -> 3) and (10 -> 9)
- OverallCond Bin rare levels,-> flag  possible drop
- BsmtFullBath  -> Bin  level (3 -> 2)
- BsmtHalfBath - >(Convert to binary or drop)
- FullBath -> Bin extremes (0 -> 1) and (4 -> 3)
- HalfBath -> Bin (2->1) or (convert to binary)
- BedroomAbvGr ->  Bin rare extremes  (0 ->1 )(6 ,7,8 -> 5)
- KitchenAbvGr  -> (Drop)
- TotRmsAbvGrd -> Bin  (14 -> 12)(2 -> 3)
- Fireplaces -> Bin (4 -> 3), consider binary
- GarageCars -> Bin (5 -> 4)
- PoolArea  - > (Drop)
- MoSold  i could split it into 3 categorys hi low mid

     -
     -
Countinus features

In [ ]:
countinus_feature = [f for f in numerical_feature if f not in years_feature and f not in discrete_feature]

print (len(countinus_feature))
print (countinus_feature)

In [ ]:
train[countinus_feature].head()

In [ ]:
# Univariate plot for numerical continous features


def cont_univ(df, continuous_features):

    fig, ax = plt.subplots(7, 3, figsize=(30, 40))

    for feature, axi in zip(continuous_features, ax.flatten()):
        sns.histplot(x=feature, data=df, ax=axi)

    plt.tight_layout()


# Bivariate for Nuumerical Continus features

def cont_biv (df , countinus_feature):


  fig,axs= plt.subplots(7, 3 , figsize=(20,30))

  for i ,axs  in zip(countinus_feature , axs.flatten()):

      sns.scatterplot(x=i , y ='SalePrice' , hue='SalePrice',data=df[countinus_feature] , ax=axs,palette='viridis_r')
      plt.xlabel(i)
      plt.ylabel('SalePrice')

      plt.title('salespric'+'-'+str(i))


# Plot the functions
cont_univ (train , countinus_feature)

cont_biv (train , countinus_feature)



In [ ]:
from scipy.stats import skew
skew_vals = train[countinus_feature].apply(skew).sort_values(ascending=False)

print(skew_vals[skew_vals.abs() > 0.75])


**univariate conclusion**
___________________________________________________________________________

For the skew i should check if its > 0.75  or i leave  

- MSSubClass    *  The numers here refeers to [60,s , 20 s ] so i  should treat it  as strings    ->  .astype(str) -> one_hot_encoding

- masvinrarea   ->  zero = 58%   ->  **Flag** + **Transform** logp1   (right skew)

- BsmtFinSF1'  ->  zero = 37%   ->  **Flag** + **Transform** logp1   (right skew)
- 'BsmtFinSF2' -> zero = 88 %  ->  Flag + drop orginal

 'BsmtUnfSF   transform (right skew)

 - openPorchsf  -> zeros + rightskew   -> flag + transform  
 - Woodicksf  ->  zeros + rightskew   -> flag + transform  

 - 1stfir1taf  ->  right skew  -> transform  
 - 2ndfir1af  ->  transform + flag

 - ScreenPorch
 _ MiscvL  drop
 - PoolArea  -> Drop
 - 3Snporch ->above 90% zeros - > Drop

________________________________________________________________________________
**CONCLUTION** :_
 ______________

 -- Flag + transform     ->      [MasVnrArea, BsmtFinSF1,
TotalBsmtSF, 2ndFlrSF, WoodDeckSF, OpenPorchSF]

 -- Flag + drop orginal ->> ['BsmtFinSF2' , 'EnclosedPorch']

  -- transform only  -> [GrLivArea, LotArea, BsmtUnfSF, 1stFlrSF, GarageArea,LotFrontage]

  --  Drop  -> [ id , 3SsnPorch, PoolArea, MiscVal]

  -- convert to Ordinal  -> [MSSubClass]


-
-

Categorical Feature

In [ ]:
# split the categorical feature to ordinal & Nominal
categorical_ordinal = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond','BsmtExposure', 'BsmtFinType1', 'BsmtFinType2','HeatingQC',
                       'KitchenQual', 'FireplaceQu','GarageFinish', 'GarageQual', 'GarageCond','PoolQC', 'Fence',
                       'LandSlope', 'LotShape', 'Utilities', 'PavedDrive']

categorical_nominal = ['MSZoning', 'Street', 'Alley', 'LandContour','LotConfig', 'Neighborhood', 'Condition1', 'Condition2',
    'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl','Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation',
    'Heating', 'CentralAir', 'Electrical', 'Functional','GarageType', 'MiscFeature', 'SaleType', 'SaleCondition'
]

In [ ]:

# PLot the Univariate and bivariate stages for Discrete and Nonminal features

def plot_univariate_and_bivariate(features, color):

    rows = len(features)
    cols = 2

    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))


    if rows == 1:
        axes = [axes]

    # univariate
    for i, feat in enumerate(features):
        counts = train[feat].value_counts().sort_index()

        counts.plot(
            kind='bar',
            ax=axes[i][0],
            color=color,
            edgecolor='black'
        )

        axes[i][0].set_title(f'{feat} Distribution')
        axes[i][0].set_xlabel(feat)
        axes[i][0].set_ylabel('Count')


        #  Bivariate
        train.boxplot(
            column='SalePrice',
            by=feat,
            ax=axes[i][1],
            grid=False
        )

        axes[i][1].set_title(f'SalePrice vs {feat}')
        axes[i][1].set_xlabel(feat)
        axes[i][1].set_ylabel('SalePrice')




    plt.suptitle("")
    plt.tight_layout()
    plt.show()


# ordinal features
plot_univariate_and_bivariate(categorical_ordinal, '#8B5CF6')

# nominal features
plot_univariate_and_bivariate(categorical_nominal, '#F59E0B')

**ordinal feature**
- ExterQual -> Po never appears -> Bin (Po->Fa) -F> Ordinal (Fa=1 < TA=2 < Gd=3 < Ex=4)
- ExterCond -> TA dominats 88% -> no ordinal trend -> Drop
- BsmtQual -> Po never appears -> Bin Ordinal (None=0 < Fa=1 < TA=2 < Gd=3 < Ex=4)
- OverallQual -> Bin (1,2->3) (10->9)
- OverallCond -> Level 5 55%  ->  Drop
- KitchenQual --> Ordinal  (Fa=1 < TA=2 < Gd=3 < Ex=4)
- HeatingQC -> Bin (Po->Fa) -> Ordinal (Fa=1 < TA=2 < Gd=3 < Ex=4)

**# note the NOne = 0 level ihave buz of the MIssing value stage above **

- BsmtExposure -> None and No dominate -> price trend  -> Ordinal (None=0 < No=1 < Mn=2 < Av=3 < Gd=4)
- BsmtFinType1 -> Ordinal (None=0 < Unf=1 < LwQ=2 < Rec=3 < BLQ=4 < ALQ=5 < GLQ=6)
- BsmtFinType2 -> None and Unf dominate 90%+ ->  weak bivariate signal -> Drop
- BsmtCond  -> No clear trend -> Drop
- FireplaceQu -> None   50% (no fireplace) -> Strong trend to houses with fireplace -> Ordinal (None=0 < Po=1 < Fa=2 < TA=3 < Gd=4 < Ex=5)
- GarageFinish  -> Ordinal (None=0 < Unf=1 < RFn=2 < Fin=3)
- GarageQual -> TA 85% --> Drop
- GarageCond -> TA  85% -> Drop
- PavedDrive -> Y  92% -> Paved vs unpaved meaningful -> Binary (Y=1, rest=0)
- LotShape -> Reg  64%, IR2,IR3 rare -> Moderate signal -> Binary (Reg=1, rest=0)
- LandSlope -> Gtl 94% -> weak signal -> Binary (Gtl=1, rest=0)
- Functional -> Typ 93% -> Typ vs deductions meaningful -> Binary (Typ=1, rest=0)
- Utilities -> AllPub 99%+ ->  Drop
________________________________________________________________________________

conclution :
________________________________________________________________________________
- [ 'ExterQual , Overall Qual'KitchenQual'  'HeatingQC'    'BsmtExposure' 'BsmtFinType1  'FireplaceQu''GarageFinish']  => ordinal

- [PavedDrive , LotShape , Functional , LandSlope ] => binary   

- ['ExterCond', 'BsmtCond', 'BsmtFinType2',
               ,Utilites 'GarageQual', 'GarageCond', 'OverallCond'] = > Drop

                

________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________

----------------------------------------------------------------------------------------------------------------------------------------------------------------

**NOMINAL FEATURES**

- MSZoning      -> Bin (C,RH->Other) -> One-hot
- Street        -> Pave dominates 97%. No signal -> Drop

- LandContour   -> Lvl dominates 85%+  Boxes overlap( see box plot) -> Binary (Lvl=1, rest=0)
- LotConfig     -> Bin (FR3->FR2) -> One-hot
- Neighborhood  -> High cardinality -> Target Encode
- Condition1    -> Bin -> (Positive, Railroad, Artery, Norm) -> One-hot
- Condition2    -> Norm dominates 97%+  -> Drop
- BldgType      -> 1Fam dominates 85%+. Others lower -> Binary (1Fam=1, rest=0)
- HouseStyle    -> price diff -> Bin rare -> One-hot
- RoofStyle     -> Gable 75% and  Hip higher price -> Binary (Hip=1, rest=0)
- RoofMatl      -> CompShg dominates 97%+ -> Drop
- Exterior1st   -> Clear price diff -> Bin rare->Other -> One-hot
- Exterior2nd   ->  Highly correlated  + redundant with (Exterior1st) -> Drop

- Foundation    -> PConc highest price -> Bin (Stone,Wood->Other) -> One-hot
- Heating       -> GasA dominates 97%. -> Binary (GasA=1, rest=0)
- CentralAir    -> Clear price diff -> Binary (Y=1, N=0)
- Electrical    -> SBrkr dominates 90%+ but  SBrkr higher price -> Binary (SBrkr=1, rest=0)
- GarageType    -> Attchd ,Detchd dominate and  BuiltIn highest -> Bin rare->Other -> One-hot

- SaleType      -> WD dominates 85%. New higher price -> Bin rare->Other -> One-hot
- SaleCondition -> Normal dominates 82%. Partial higher price -> Bin rare->Other -> One-hot

________________________________________________________________________________
________________________________________________________________________________
Conclusion :
_________________________________________________________________________________

- onehot = ['MSZoning', 'LotConfig', 'HouseStyle', 'Condition1',
           'Exterior1st', 'Foundation', 'GarageType',
           'SaleType', 'SaleCondition']

- target = ['Neighborhood']

- binary = ['LandContour', 'BldgType', 'RoofStyle',
          'Heating', 'CentralAir', 'Electrical']

- drop = ['Street', 'Condition2', 'RoofMatl',
        'Exterior2nd', 'Alley', 'MasVnrType', 'MiscFeature']

**MultiVariate**

In [ ]:
# Select top correlated features with target
corr = train[numerical_feature].corr()['SalePrice']
top_features = corr.abs().sort_values(ascending=False).head(6).index.tolist()

sns.pairplot(train[top_features])

In [ ]:
# corroleation between variables
plt.subplots(figsize = (30,20))
mask = np.zeros_like(train[numerical_feature].corr(), dtype=np.bool)
mask[np.triu_indices_from(mask)] = True
#Plotting heatmap
sns.heatmap(train[numerical_feature].corr(), cmap=sns.diverging_palette(20, 220, n=200), mask = mask, annot=True, center = 0)

GarageArea & GarageCars .88 |
HIGH Corrlation between GrLivArea & TotRmsAbva .83  |  TotalBsmtsf & 1stlrsf .82



In [ ]:

sns.histplot(x ='SalePrice' , data = train , color ='skyblue' )


**Saleprice is not normaly distrbuted its  Right skewed **

In [ ]:

print (f"The skew of Target  {train['SalePrice'].skew()}")
print (f"The Kurtosis of Target  {train['SalePrice'].kurtosis()}")

In [ ]:
# reomve Skewness

train['SalePrice'] = np.log1p(train['SalePrice'])


sns.histplot(x = 'SalePrice' , data = train)

# Feature Engineering

The raw data is messy and not in a shape that models can learn from well.  
So I'll go through :

- **Add features** — TotalSF, TotalBath, HasGarage ,  things I think buyers actually care about
- **Drop** columns that are mostly missing or just not useful  
- **Impute** the missing values — None for absent features, 0 for areas, mode for categoricals  
- **Flag** features that have a lot of zeros sometimes having vs not having 
- **Transform** skewed features with log1p to reduce the effect of extreme values  
- **Remove outliers** — a couple of massive houses sold for way below market, they'll confuse the model  
- **Bin** rare categories together so the model doesn't overfit to them  
- **Encode** categoricals — ordinal where there's a natural order, one-hot for nominals  

In [ ]:
def add_features(df):
    # Total square footage across all floors [[ basement + 1st + 2nd floor]]
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    
    # Total number of bathrooms in the house (half bath = 0.5 because it has no shower)
    df['TotalBath'] = df['FullBath'] + 0.5*df['HalfBath'] + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath']
    # 0 if there is no garage 
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)


    df['TotalPorchSF'] = (
        df['OpenPorchSF'] + df['EnclosedPorch'] + df['WoodDeckSF'] + df.get('ScreenPorch', 0) +
        df.get('3SsnPorch', 0)
    )

    # quality × size interaction 
    df['QualSF'] = df['OverallQual'] * df['GrLivArea']

    # NEW: bath squared (small count, no transform needed)
    df['TotalBath2'] = df['TotalBath'] ** 2
    return df   

train = add_features(train)
test = add_features(test)

**DROP FEATURES**


In [ ]:
from logging import error
def drop_features(df):

  columns_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType','PoolArea' ,'KitchenAbvGr' ,'Id' ,
                            '3SsnPorch', 'PoolArea', 'MiscVal' ,'ExterCond', 'BsmtCond', 'BsmtFinType2', 'Utilities' ,
                            'GarageQual', 'GarageCond', 'OverallCond' ,'Street', 'Condition2', 'RoofMatl','Exterior2nd',
                            # years_features
                            'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'YrSold']


  for col in columns_to_drop:
    if col not in df.columns:
      print(col)

  return df.drop(columns = columns_to_drop ,errors='ignore')

In [ ]:
train = drop_features(train)
test = drop_features(test)

print (train.shape)
print (test.shape)

**IMPUTE**

In [ ]:
def impute(df):
    df = df.copy()

    # 1. absence missing
    none_impute = [
        'GarageFinish', 'GarageType', 'BsmtQual',
        'BsmtExposure', 'BsmtFinType1', 'FireplaceQu'
    ]


    zero_impute = [
        'MasVnrArea', 'BsmtFullBath',
        'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
        'TotalBsmtSF', 'GarageCars', 'GarageArea',
        'BsmtHalfBath'
    ]


    mode_impute = [
        'Electrical', 'MSZoning', 'Exterior1st',
        'KitchenQual', 'Functional', 'SaleType'
    ]

    #  NONE imputation
    for col in none_impute:
        if col in df.columns:
            df[col] = df[col].fillna('NONE')

    #  Zero imputation
    for col in zero_impute:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # MODE imptation
    for col in mode_impute:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])


    if 'LotFrontage' in df.columns and 'Neighborhood' in df.columns:
        df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
            lambda x: x.fillna(x.median())
        )
        df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())

    return df

train = impute(train)
test = impute(test)

In [ ]:
print("train:", train.isnull().sum().sum())
print("TEST:", test.isnull().sum().sum())


print (train.isnull().sum()[train.isnull().sum()>0])
print (test.isnull().sum()[test.isnull().sum()>0])

**FLAG FEATURES**

In [ ]:
# Flage the needed features
def flag(df):

  columns_flag =  ['MasVnrArea', 'BsmtFinSF1', 'TotalBsmtSF', '2ndFlrSF', 'WoodDeckSF', 'OpenPorchSF' , 'TotalPorchSF']
  drop_after_flag = ['BsmtFinSF2' , 'EnclosedPorch']


  for col in columns_flag :
    df[col+"_flag"] = (df[col]>0).astype(int)

  for col in drop_after_flag:
    df[col+"flag"] = (df[col]>0).astype(int)
    df = df.drop(columns =[col])

  return df

train = flag(train)
test = flag(test)

**TRANSFORM**


In [ ]:
# transform
def transform(df):
    log_cols = ['GrLivArea', 'LotArea', 'LotFrontage', 'BsmtUnfSF',
                '1stFlrSF', 'GarageArea', 'MasVnrArea', 'BsmtFinSF1',
                'TotalBsmtSF', '2ndFlrSF', 'WoodDeckSF', 'OpenPorchSF',
                'HouseAge', 'remodAge', 'garageAge','TotalPorchSF', 'QualSF']
    for col in log_cols:
        if col in df.columns:
            df[col] = df[col].clip(lower=0)
            df[col] = np.log1p(df[col])
    return df

train = transform(train)
test  = transform(test)

**OUTLIERs TREATMENT**

**#note** iwill apply the **isolation forest** after the data spliting  operation to avoid Data Leakage


In [ ]:
# Run this AFTER calling transform(), not before
outlier_idx = train[
    (train['GrLivArea'] > np.log1p(4000)) &   
    (train['SalePrice'] < np.log1p(200000))    
].index

train = train.drop(outlier_idx)
print(f"REMOVED {len(outlier_idx)} ROWS")

**BIN**

In [ ]:
# bin discrete values
def bin_discrete(df):

    df['OverallQual'] = df['OverallQual'].replace({1: 3, 2: 3, 10: 9})

    df['BsmtFullBath'] = df['BsmtFullBath'].replace({3: 2})

    df['BsmtHalfBath'] = (df['BsmtHalfBath'] > 0).astype(int)

    df['FullBath'] = df['FullBath'].replace({0: 1, 4: 3})

    df['HalfBath'] = (df['HalfBath'] > 0).astype(int)

    df['BedroomAbvGr'] = df['BedroomAbvGr'].replace({0: 1, 6: 5, 7: 5, 8: 5})

    df['TotRmsAbvGrd'] = df['TotRmsAbvGrd'].replace({2: 3, 14: 12})

    df['Fireplaces'] = df['Fireplaces'].replace({4: 3})

    df['GarageCars'] = df['GarageCars'].replace({5: 4})

    df['MoSold']  = df['MoSold'].apply (lambda x : 'high' if x in [4,5,6,7] else ('mid' if x in [3,8,10 ,11] else 'low'))

    return df


train = bin_discrete(train)
test = bin_discrete(test)

In [ ]:
print(set(train.columns) - set(test.columns))
print(set(test.columns) - set(train.columns))

**Ordinal ENC**


In [ ]:
def ordinal_enc(df):

    # ExterQual
    df['ExterQual'] = df['ExterQual'].replace({'Po': 'Fa'})
    df['ExterQual'] = df['ExterQual'].map({'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4}).fillna(2)

    # BsmtQual
    df['BsmtQual'] = df['BsmtQual'].map({'NONE': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4}).fillna(0)

    # KitchenQual — 'NONE' possible from impute
    df['KitchenQual'] = df['KitchenQual'].replace({'NONE': 'TA'})
    df['KitchenQual'] = df['KitchenQual'].map({'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4}).fillna(2)

    # HeatingQC
    df['HeatingQC'] = df['HeatingQC'].replace({'Po': 'Fa'})
    df['HeatingQC'] = df['HeatingQC'].map({'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4}).fillna(2)

    # BsmtExposure
    df['BsmtExposure'] = df['BsmtExposure'].map({'NONE': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}).fillna(0)

    # BsmtFinType1
    df['BsmtFinType1'] = df['BsmtFinType1'].map({'NONE': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}).fillna(0)

    # FireplaceQu
    df['FireplaceQu'] = df['FireplaceQu'].map({'NONE': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}).fillna(0)

    # GarageFinish
    df['GarageFinish'] = df['GarageFinish'].map({'NONE': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}).fillna(0)

    # Binary
    df['PavedDrive'] = (df['PavedDrive'] == 'Y').astype(int)
    df['LotShape']   = (df['LotShape']   == 'Reg').astype(int)
    df['LandSlope']  = (df['LandSlope']  == 'Gtl').astype(int)
    df['Functional'] = (df['Functional'] == 'Typ').astype(int)

    return df


# compined them to avoid mismatch
train_size = train.shape[0]

combined = pd.concat([train.drop(columns=['SalePrice']), test], axis=0).reset_index(drop=True)
combined  = ordinal_enc(combined)

train_encoded = combined.iloc[:train_size].copy()
test          = combined.iloc[train_size:].copy()

train_encoded['SalePrice'] = train['SalePrice'].values
train = train_encoded



**ENCODE NoMINAL**

In [ ]:
def encode_nominal(df):

    # Binary encode
    df['LandContour'] = (df['LandContour'] == 'Lvl').astype(int)
    df['BldgType']    = (df['BldgType']    == '1Fam').astype(int)
    df['RoofStyle']   = (df['RoofStyle']   == 'Hip').astype(int)
    df['Heating']     = (df['Heating']     == 'GasA').astype(int)
    df['CentralAir']  = (df['CentralAir']  == 'Y').astype(int)
    df['Electrical']  = (df['Electrical']  == 'SBrkr').astype(int)

    # Bin rare levels
    df['MSZoning']   = df['MSZoning'].replace({'C (all)': 'Other', 'RH': 'Other'})
    df['LotConfig']  = df['LotConfig'].replace({'FR3': 'FR2'})
    df['Condition1'] = df['Condition1'].replace({'PosN': 'Positive', 'PosA': 'Positive',
                                                 'RRNn': 'Railroad', 'RRAn': 'Railroad',
                                                 'RRNe': 'Railroad', 'RRAe': 'Railroad'})
    df['HouseStyle'] = df['HouseStyle'].replace({'1.5Unf': 'Other', '2.5Unf': 'Other', '2.5Fin': 'Other'})

    keep = ['VinylSd', 'HdBoard', 'MetalSd', 'Wd Sdng', 'Plywood', 'CemntBd', 'BrkFace', 'WdShing']
    df['Exterior1st'] = df['Exterior1st'].apply(lambda x: x if x in keep else 'Other')

    df['Foundation'] = df['Foundation'].replace({'Stone': 'Other', 'Wood': 'Other'})

    keep_garage = ['Attchd', 'Detchd', 'BuiltIn', 'Basment', 'NONE']
    df['GarageType'] = df['GarageType'].apply(lambda x: x if x in keep_garage else 'Other')

    keep = ['WD', 'New', 'COD']
    df['SaleType'] = df['SaleType'].apply(lambda x: x if x in keep else 'Other')

    keep = ['Normal', 'Partial', 'Abnorml']
    df['SaleCondition'] = df['SaleCondition'].apply(lambda x: x if x in keep else 'Other')

    df['MSSubClass'] = df['MSSubClass'].astype(str)
    keep = ['20', '30', '50', '60', '70', '80', '90', '120', '160']
    df['MSSubClass'] = df['MSSubClass'].apply(lambda x: x if x in keep else 'Other')

    # One-hot
    ohe_cols = ['MSZoning', 'LotConfig', 'Condition1', 'HouseStyle',
                'Exterior1st', 'Foundation', 'GarageType', 'SaleType',
                'SaleCondition', 'MoSold', 'MSSubClass']

    df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

    return df


# encode both toghether to garatiy identical columns

train_size = train.shape[0]

combined = pd.concat([train.drop(columns=['SalePrice']), test], axis=0).reset_index(drop=True)
combined  = encode_nominal(combined)

# return them back
train_encoded = combined.iloc[:train_size].copy()
test          = combined.iloc[train_size:].copy()

train_encoded['SalePrice'] = train['SalePrice'].values
train = train_encoded

print(train.shape)
print(test.shape)

In [ ]:
# check what is different
data_cols = set(train.drop(columns=['SalePrice']).columns)
test_cols  = set(test.columns)

print("In train but not test:", data_cols - test_cols)
print("In test but not train:", test_cols - data_cols)

# Align test to train
test = test.reindex(columns=train.drop(columns=['SalePrice']).columns, fill_value=0)

print(train.shape)
print(test.shape)

**SPLIT THE DATA**

In [ ]:
from numpy.random.mtrand import random
X = train.drop(columns=['SalePrice'])
Y = train['SalePrice']

X_train , X_val , y_train , y_val = train_test_split(X ,Y , test_size = 0.2 , random_state = 42)


In [ ]:
# TARGET ENCODE 

neighborhood_means = y_train.groupby(X_train['Neighborhood']).mean()

X_train['Neighborhood'] = X_train['Neighborhood'].map(neighborhood_means)
X_val['Neighborhood']   = X_val['Neighborhood'].map(neighborhood_means)
test['Neighborhood']    = test['Neighborhood'].map(neighborhood_means)

# Handle unseen neighborhood by filling it with the global mean 
global_mean = y_train.mean()
X_train['Neighborhood'] = X_train['Neighborhood'].fillna(global_mean)
X_val['Neighborhood']   = X_val['Neighborhood'].fillna(global_mean)
test['Neighborhood']    = test['Neighborhood'].fillna(global_mean)

# Feature Selection

When working with data, more features doesn’t always mean better performance. having too many irrelevant or redundant features can hurt the model and make it more complex than necessary

To improve the quality of the data, I will apply :

- **Variance filter** – remove features that stay almost constant, since they don’t provide meaningful information
- **Correlation filter** – if two features are very similar, I’ll keep one and drop the other to avoid redundancy
- **RFE (Recursive Feature Elimination)** – use a Random Forest model to identify the most important features and eliminate the less useful ones

**Isolation Forest step** 

In [ ]:
from sklearn.ensemble import IsolationForest

iso  = IsolationForest(contamination=0.02, random_state=42)
mask = iso.fit_predict(X_train) == -1   

# use iloc to avoid index mismatch
X_train = X_train.iloc[~mask].reset_index(drop=True)
y_train = y_train.iloc[~mask].reset_index(drop=True)

print(f"Removed {mask.sum()} outliers — {len(X_train)} rows remaining")

**VARIANCE**


In [ ]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)

selector.fit(X_train)

low_var_cols = X_train.columns[~selector.get_support()].tolist()

print('low variance columns are ',low_var_cols)

variance = X_train.var()
print (variance[low_var_cols].sort_values(ascending = False))


In [ ]:
print(X_train[low_var_cols].describe())

# drop low variance colums
X_train = X_train.drop(columns = low_var_cols)
X_val = X_val.drop(columns = low_var_cols)
test = test.drop(columns = low_var_cols)

**CORRELATION FEATURES**

In [ ]:
# GarageArea / GarageCars -> 0.88
# GrLivArea / TotRmsAbvGrd -> 0.83 ->  TotRmsAbvGrd already droped
# TotalBsmtSF / 1stFlrSF -> 0.82  -> drop 1stFlrSF

corr_features = ['GarageCars'  , '1stFlrSF']

X_train = X_train.drop(columns = corr_features)
X_val = X_val.drop(columns = corr_features)
test = test.drop(columns = corr_features)

**SCALE THE DATA**

In [ ]:
from sklearn.preprocessing import RobustScaler

# Convert back to DataFrame if scaling already ran once
if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train, columns=features_names)
    X_val   = pd.DataFrame(X_val,   columns=features_names)
    test    = pd.DataFrame(test,     columns=features_names)

X_train = X_train.fillna(0)
X_val   = X_val.fillna(0)
test    = test.fillna(0)

features_names = X_train.columns.tolist()

scale          = RobustScaler()
X_train_scaled = scale.fit_transform(X_train)
X_val_scaled   = scale.transform(X_val)
test_scaled    = scale.transform(test.reindex(columns=features_names, fill_value=0))

X_train = pd.DataFrame(X_train_scaled, columns=features_names)
X_val   = pd.DataFrame(X_val_scaled,   columns=features_names)
test    = pd.DataFrame(test_scaled,    columns=features_names)

In [ ]:
# from sklearn.preprocessing import RobustScaler


# X_train = X_train.fillna(0)
# X_val = X_val.fillna(0)
# test = test.fillna(0)
# #save the columns names
# features_names = X.columns

# # Scale data
# scale = RobustScaler()

# X_train =scale.fit_transform(X_train)
# X_val = scale.transform(X_val)

# # add to test columns features names
# test = test.reindex(columns=features_names, fill_value=0)
# test = scale.transform (test)


# # return it back to Dataframe

# X_train = pd.DataFrame(X_train , columns = features_names)
# X_val = pd.DataFrame(X_val , columns = features_names)
# test = pd.DataFrame(test , columns = features_names)

In [ ]:
print (set(X_train.columns)-set(test.columns))
print (set(test.columns)-set(X_train.columns))



**RFE**

In [ ]:
# from sklearn.feature_selection import RFECV
# from sklearn.ensemble import RandomForestRegressor

# rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# rfe = RFECV(
#     estimator=rf,
#     step=1,              # remove 1 fet at a time
#     cv=5,
#     scoring='neg_mean_squared_error',
#     n_jobs=-1
# )

# rfe.fit(X_train, y_train)

# print(f"best number of features: {rfe.n_features_}")

# # Apply The selection
# selected_cols = X_train.columns[rfe.support_].tolist()

# X_train_rfe = X_train[selected_cols].copy()
# X_val_rfe   = X_val[selected_cols].copy()
# test_rfe    = test[selected_cols].copy()

# print("Selected_features:", selected_cols)
# print("Final_shape:", X_train.shape)

# MODELING

In this step, I build and compare different models. I start with simple models and then move to more advanced ones.

###   #  1. Cross Validation Setup
###   #  2. Base Models Comparison
###   #  3. Hyperparameter Tuning
###   #  4. Stacking
###   #  5. BLENDING

###   #  6. Final Evaluation
###   #  7. SAVING MODELS 
###   #  8. Submission

I tune each model using GridSearchCV and evaluate them with 5-fold cross-validation. This helps me get reliable results and avoid depending on one data split.

**Cross Validation Setup**

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

**EVALUATE MODEL FUNCTION**

In [ ]:
from sklearn.metrics import root_mean_squared_error
from sklearn.base import clone

from sklearn.model_selection import GridSearchCV
"""
evaluate the performance of the model by 
    - applying the corss validation to check how to model will generlize (how it will work with unseen data)
    - calc the RMSE For each model 
    - Check The overfiting by calculating the diffrence beween the train_rmse and val_rmse 

"""
def evaluate_model(name, model, X_train, y_train, X_val, y_val, kf):
    cv_scores = []

    for train_idx, val_idx in kf.split(X_train):
        X_tr = X_train.iloc[train_idx]
        y_tr = y_train.iloc[train_idx]
        X_vl = X_train.iloc[val_idx]
        y_vl = y_train.iloc[val_idx]

        model_clone = clone(model)
        model_clone.fit(X_tr, y_tr)
        preds = model_clone.predict(X_vl)   
        rmse  = root_mean_squared_error(y_vl, preds)
        cv_scores.append(rmse)

    cv_rmse = np.mean(cv_scores)           


    model.fit(X_train, y_train)

    train_rmse = root_mean_squared_error(y_train, model.predict(X_train))
    val_rmse   = root_mean_squared_error(y_val,   model.predict(X_val))

    print(f"{name}:")
    print(f"  CV RMSE    : {cv_rmse:.4f} ± {np.std(cv_scores):.4f}")
    print(f"  Train RMSE : {train_rmse:.4f}")
    print(f"  Val RMSE   : {val_rmse:.4f}")

    gap = val_rmse - train_rmse
    if gap > 0.02:
        print(f"  Overfitting — gap: {gap:.4f}")
    else:
        print(f"  Healthy — gap: {gap:.4f}")

    return cv_rmse, val_rmse, model

### Tune models
  - linear regression
  - Lasso
  - Ridge
  - Xgboost
  - Lightgbm
  - Random Forest
  - GBM
  - Stacking
    
   

In [ ]:
from sklearn.linear_model import LinearRegression


model = LinearRegression()

model.fit(X_train ,y_train)

predict = model.predict(X_val)

print ("the  rmse is : "  , root_mean_squared_error(y_val , predict))

In [ ]:
# lasso model

from sklearn.linear_model import Lasso


lasso_param = {
    "alpha"     : [0.0001, 0.001, 0.01, 0.1, 1,10 ,100],
    "max_iter"  : [1000, 5000, 10000]

}


"""
USE GridSearchCV to find the best hyperparameters 
"""
lasso_grid = GridSearchCV(
    Lasso(random_state = 42),
    lasso_param ,
    cv = 5 ,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
lasso_grid.fit(X_train ,y_train)

best_lasso = lasso_grid.best_estimator_
best_lasso_params = lasso_grid.best_params_

print(f"Best Lasso params: {best_lasso_params}")


# Evaluate 
lasso_cv_rmse, lasso_val_rmse, best_lasso = evaluate_model(
    'Lasso', best_lasso, X_train, y_train, X_val, y_val, kf
)

In [ ]:
# Ridge Model 

from sklearn.linear_model import Ridge


ridge_params = {
    "alpha": [0.1 ,0.5, 1, 10, 50, 100]
}

ridge_grid = GridSearchCV(
    Ridge(random_state=42),
    ridge_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

best_ridge = ridge_grid.best_estimator_
best_ridge_params = ridge_grid.best_params_

print(f"Best Ridge params: {best_ridge_params}")

ridge_cv_rmse, ridge_val_rmse, best_ridge = evaluate_model(
    "Ridge", best_ridge, X_train, y_train, X_val, y_val, kf
)

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

xgb_params = {
    "max_depth"        : [2, 3, 4],
    "learning_rate"    : [0.01, 0.03, 0.05],
    "subsample"        : [0.6, 0.7, 0.8],
    "colsample_bytree" : [0.6, 0.7, 0.8],
    "reg_lambda"       : [1, 5, 10],
    "min_child_weight" : [3, 5, 7],
}

xgb_search = RandomizedSearchCV(
    XGBRegressor(n_estimators=300, random_state=42),
    xgb_params,
    n_iter      = 30,         
    cv          = 5,
    scoring     = 'neg_mean_squared_error',
    random_state= 42,
    n_jobs      = -1
)

xgb_search.fit(X_train, y_train)

print(f"Best XGBoost params: {xgb_search.best_params_}")

# final model 
best_xgb = XGBRegressor(
    **xgb_search.best_params_,
    n_estimators = 300,
    random_state = 42
)

xgb_cv_rmse, xgb_val_rmse, best_xgb = evaluate_model(
    'XGBoost', best_xgb, X_train, y_train, X_val, y_val, kf
)

In [ ]:
import lightgbm as lgb

lgb_params = {
    'num_leaves':       [15, 31, 63, 127],
    'learning_rate':    [0.01, 0.03, 0.05],
    'min_child_samples':[10, 20, 30],
    'subsample':        [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha':        [0.0, 0.1, 0.5],
    'reg_lambda':       [1, 5, 10],
}

lgb_search = RandomizedSearchCV(
    lgb.LGBMRegressor(n_estimators=500, random_state=42, verbose=-1),
    lgb_params,
    n_iter=40,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)
lgb_search.fit(X_train, y_train)
best_lgb = lgb_search.best_estimator_
print(f"Best LightGBM params: {lgb_search.best_params_}")

lgb_cv_rmse, lgb_val_rmse, best_lgb = evaluate_model(
    'LightGBM', best_lgb, X_train, y_train, X_val, y_val, kf
)

In [ ]:
from sklearn.linear_model import ElasticNet

en_params = {
    'alpha':    [0.0001, 0.001, 0.01, 0.1],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
    'max_iter': [5000]
}

en_grid = GridSearchCV(
    ElasticNet(random_state=42),
    en_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
en_grid.fit(X_train, y_train)
best_en = en_grid.best_estimator_
print(f"Best ElasticNet params: {en_grid.best_params_}")

en_cv_rmse, en_val_rmse, best_en = evaluate_model(
    'ElasticNet', best_en, X_train, y_train, X_val, y_val, kf
)

In [ ]:
# Catboost

from catboost import CatBoostRegressor

best_cat = CatBoostRegressor(
    iterations    = 1000,
    learning_rate = 0.03,
    depth         = 6,
    l2_leaf_reg   = 3,
    random_seed   = 42,
    eval_metric   = 'RMSE',
    verbose       = False
)

best_cat.fit(
    X_train, y_train,
    eval_set            = (X_val, y_val),
    early_stopping_rounds = 50
)

cat_cv_rmse, cat_val_rmse, best_cat = evaluate_model(
    'CatBoost', best_cat, X_train, y_train, X_val, y_val, kf
)

In [ ]:
# # Random Forest 

# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import GridSearchCV

# rf_params = {
#     "n_estimators": [100, 300, 500],
#     "max_depth": [None, 10, 20, 30],
#     "min_samples_split": [2, 5, 10]
# }

# rf_grid = GridSearchCV(
#     RandomForestRegressor(random_state=42, n_jobs=-1),
#     rf_params,
#     cv=5,
#     scoring='neg_mean_squared_error',
#     n_jobs=-1
# )

# rf_grid.fit(X_train, y_train)

# best_rf = rf_grid.best_estimator_
# best_rf_params = rf_grid.best_params_

# print(f"Best RF params: {best_rf_params}")

# rf_cv_rmse, rf_val_rmse, best_rf = evaluate_model(
#     "RandomForest", best_rf, X_train, y_train, X_val, y_val, kf
# )

In [ ]:
# Gbm regressor 

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

gbm_params = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.001,0.01, 0.05, 0.1 ,1],
    "max_depth": [2,3, 4, 5]
}

gbm_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gbm_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

gbm_grid.fit(X_train, y_train)

best_gbm = gbm_grid.best_estimator_
best_gbm_params = gbm_grid.best_params_

print(f"Best GBM params: {best_gbm_params}")

gbm_cv_rmse, gbm_val_rmse, best_gbm = evaluate_model(
    "GBM", best_gbm, X_train, y_train, X_val, y_val, kf
)

### Stacking 

In [ ]:
from sklearn.ensemble import StackingRegressor

# Base models 
base_models = [
    ('ridge', best_ridge),
    ('elastic', best_en),
    ('lasso', best_lasso),
    ('xgb', best_xgb),
    # ('gbm', best_gbm),
    ('cat', best_cat),
    # ('lightgbm' , best_lgb)
]

# Meta model 
meta_model = Ridge()

# Stacking model
stack_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

# train stacking
stack_model.fit(X_train, y_train)

# evaluation 
stack_cv_rmse, stack_val_rmse, stack_model = evaluate_model(
    "Stacking",
    stack_model,
    X_train, y_train,
    X_val, y_val,
    kf
)

### The RMSE Plot  


In [ ]:
results = {
    "Ridge": ridge_val_rmse,
    "ElasticNet": en_val_rmse,
    "Catboost": cat_val_rmse,
    "Lasso": lasso_val_rmse,
    # "RandomForest": rf_val_rmse,
    # "GBM": gbm_val_rmse,
    "XGBoost": xgb_val_rmse ,
    "LightGbm" : lgb_val_rmse,
    'Stacking': stack_val_rmse,
    # 'Blend' : blend_val_rmse
}


models = list(results.keys())
scores = list(results.values())


plt.figure(figsize=(8,5))
plt.bar(models, scores)

plt.title("Model Comparison (Validation RMSE)  Less is better ")
plt.ylabel("RMSE")
plt.xticks(rotation=45)

plt.show()


In [ ]:

cv_scores = {
    "Ridge": ridge_cv_rmse,
    "Lasso": lasso_cv_rmse,
    "ElasticNet": en_cv_rmse,
    "Catboost" : cat_cv_rmse,
    # "GBM": gbm_cv_rmse,
    "XGB": xgb_cv_rmse ,
    "LightGbm" : lgb_cv_rmse,
    "Stacking": stack_cv_rmse,
    # "Blend"      : blend_val_rmse  
}



models = list(cv_scores.keys())
cv = list(cv_scores.values())
val = list(results.values())

x = np.arange(len(models))

plt.figure(figsize=(10,5))
plt.bar(x - 0.2, cv, 0.4, label="CV RMSE")
plt.bar(x + 0.2, val, 0.4, label="Validation RMSE")

plt.xticks(x, models, rotation=45)
plt.ylabel("RMSE")
plt.title("CV vs Validation RMSE")
plt.legend()

plt.show()

**RESDUAL ANALYSIS**

In [ ]:
# resdual = y_val - stack_model.predict(y_val)

# plt.scatter (stack_model.predict(y_val) , resdual)
# plt.axhline(0)
# plt.xlabel("predicted")
# plt.ylabel("residuals")
# plt.title("residuals vs predicted")
# plt.show()

## save models and features 

In [ ]:

os.makedirs('/kaggle/working/models', exist_ok=True)

joblib.dump(best_xgb,    '/kaggle/working/models/best_xgb.pkl')
joblib.dump(best_en,    '/kaggle/working/models/best_en.pkl')
joblib.dump(best_lgb,    '/kaggle/working/models/best_lgb.pkl')
# joblib.dump(best_rf,     '/kaggle/working/models/best_rf.pkl')
joblib.dump(best_gbm,    '/kaggle/working/models/best_gbm.pkl')
joblib.dump(best_lasso,  '/kaggle/working/models/best_lasso.pkl')
joblib.dump(best_ridge,  '/kaggle/working/models/best_ridge.pkl')
joblib.dump(stack_model, '/kaggle/working/models/stack_model.pkl')
joblib.dump(scale,       '/kaggle/working/models/scaler.pkl')
joblib.dump(rfe, '/kaggle/working/models/rfe.pkl')
print(" models saved")

In [ ]:
import shutil

# zip the  models folder into one file
shutil.make_archive('/kaggle/working/house-price-models', 'zip', '/kaggle/working/models')
print("completed")

**reuse The models and features** 

In [ ]:
# best_xgb    = joblib.load('/kaggle/input/datasets/abdalaziz23/models/best_xgb.pkl')
# best_lgb    = joblib.load('/kaggle/input/datasets/abdalaziz23/models/best_lgb.pkl')
# best_rf     = joblib.load('/kaggle/input/datasets/abdalaziz23/models/best_rf.pkl')
# best_gbm    = joblib.load('/kaggle/input/datasets/abdalaziz23/models/best_gbm.pkl')
# best_lasso  = joblib.load('/kaggle/input/datasets/abdalaziz23/models/best_lasso.pkl')
# best_ridge  = joblib.load('/kaggle/input/datasets/abdalaziz23/models/best_ridge.pkl')
# stack_model = joblib.load('/kaggle/input/datasets/abdalaziz23/models/stack_model.pkl')
# scale       = joblib.load('/kaggle/input/datasets/abdalaziz23/models/scaler.pkl')
# rfe         = joblib.load('/kaggle/input/datasets/abdalaziz23/models/rfe.pkl')

# print(" models loaded ")

In [ ]:
X_full = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_full = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

# retrain each model on full data
best_xgb.fit(X_full, y_full)
best_lgb.fit(X_full, y_full)
best_cat.fit(X_full, y_full)
best_ridge.fit(X_full, y_full)
best_lasso.fit(X_full, y_full)
best_en.fit(X_full, y_full)
stack_model.fit(X_full, y_full)


# submission 

In [ ]:




final_predictions = stack_model.predict(test)



# Reverse the log transform
final_predictions = np.expm1(final_predictions)

print(f"Min  price: ${final_predictions.min():,.0f}")
print(f"Max  price: ${final_predictions.max():,.0f}")
print(f"Mean price: ${final_predictions.mean():,.0f}")

submission = pd.read_csv(f"{path}/sample_submission.csv")
submission['SalePrice'] = final_predictions
submission.to_csv('/kaggle/working/submission.csv', index=False)

In [ ]:
submission.head()